# 🧠 Notebook 15: Mixture of Experts (MoE)

Scaling model capacity without proportionally scaling compute — the key idea behind modern LLMs like Mixtral, DeepSeek-V3, and Gemma 4.

**Prerequisites:** Notebooks 06 (Transformer Architecture), 09 (Modern Architectures)

**What you'll learn:**
- 💡 Why sparse MoE beats dense models at scale
- 🎯 Complete math derivation of Top-K routing
- ⚡ How MoE achieves more capacity with less compute per token
- ⚠️ The expert collapse problem and how load balancing solves it

## The Problem: Dense Models Hit a Wall

In a standard (dense) transformer, **every token activates every parameter** in every layer. If you want a smarter model, you make it bigger — but that means every token now costs more compute:

| Model | Parameters | Active per Token | FLOPs per Token |
|-------|-----------|-----------------|-----------------|
| LLaMA 7B (dense) | 7B | 7B (100%) | ~14 TFLOPs |
| LLaMA 70B (dense) | 70B | 70B (100%) | ~140 TFLOPs |
| Mixtral 8x7B (MoE) | 47B | ~13B (28%) | ~26 TFLOPs |
| DeepSeek-V3 (MoE) | 671B | ~37B (5.5%) | ~74 TFLOPs |

💡 **Key insight:** MoE decouples model *capacity* (total parameters) from model *cost* (compute per token). You get the knowledge of a 671B model at the cost of a ~37B model.

⚠️ **The catch:** You need a smart *routing* mechanism to decide which experts handle which tokens — and if routing goes wrong, some experts get all the work while others sit idle ("expert collapse").

## MoE at a Glance

In a standard transformer block, the FFN processes every token identically:

```
token → Attention → FFN → output
```

In an MoE block, the single FFN is replaced by **N expert FFNs** plus a **router** that picks the best K experts per token:

```
token → Attention → Router → [Expert_i, Expert_j] → weighted sum → output
```

Each expert is a full FFN (same architecture), but each token only activates K of them. This is **conditional computation** — the model learns *which* parameters to use for *which* inputs.

🎯 **Interview tip:** MoE is not an approximation. Each token gets the *exact* output of its selected experts. The sparsity is in *which* experts fire, not in the quality of computation.

## Setup

In [ ]:
import mlx.core as mx
import mlx.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
import math

# Import MoE components from utils module
from utils.moe import MoEConfig, MoERouter, ExpertChoiceRouter, HashRouter, ExpertFFN, MoEBlock, top_k

mx.random.seed(42)
print(f"MLX version: {mx.__version__}")
print("✅ Ready for MoE!")

---

## Math Derivation: Top-K Routing

Following the pedagogical pattern: **math first**, then code, then visualization.

The router is the brain of MoE — it decides which experts handle each token. Let's derive Top-K routing step by step.

### Step 1: Gating Logits

Given an input token embedding $\mathbf{x} \in \mathbb{R}^{d_{\text{model}}}$, the router computes a score for each expert using a learned weight matrix $\mathbf{W}_{\text{gate}} \in \mathbb{R}^{d_{\text{model}} \times N}$ where $N$ is the number of experts:

$$\mathbf{g} = \mathbf{x} \cdot \mathbf{W}_{\text{gate}}$$

**Shape derivation** for a batch of sequences:
- Input: $\mathbf{X} \in \mathbb{R}^{B \times S \times d_{\text{model}}}$ (batch × sequence × model dim)
- Gate weights: $\mathbf{W}_{\text{gate}} \in \mathbb{R}^{d_{\text{model}} \times N}$
- Router logits: $\mathbf{G} = \mathbf{X} \cdot \mathbf{W}_{\text{gate}} \in \mathbb{R}^{B \times S \times N}$

Each of the $B \times S$ tokens gets $N$ scores — one per expert.

💡 **Insight:** This is just a linear projection. The router is the simplest possible neural network: a single matrix multiply with no bias and no activation. The magic is in what happens *after*.

### Step 2: Top-K Selection

We don't want to run *all* $N$ experts for every token — that would defeat the purpose. Instead, we select the top $K$ experts with the highest gating logits:

$$\text{top\_k\_logits}, \text{top\_k\_indices} = \text{TopK}(\mathbf{g}, k)$$

- $\text{top\_k\_logits} \in \mathbb{R}^{B \times S \times K}$ — the $K$ highest scores per token
- $\text{top\_k\_indices} \in \mathbb{Z}^{B \times S \times K}$ — which experts were selected

For example, with $N = 8$ experts and $K = 2$:
- Token "The" might route to experts [2, 5] with logits [3.1, 2.8]
- Token "cat" might route to experts [0, 7] with logits [4.2, 1.9]

⚡ **Performance:** Only $K$ out of $N$ expert FFNs run per token. With $K = 2$ and $N = 8$, you use 25% of the expert compute. Mixtral uses $K = 2, N = 8$. DeepSeek-V3 uses $K = 8, N = 256$.

### Step 3: Weight Normalization via Softmax

The selected logits are converted to proper weights using softmax **over the selected experts only** (not all $N$):

$$\mathbf{w} = \text{softmax}(\text{top\_k\_logits}) = \frac{\exp(\text{top\_k\_logits}_i)}{\sum_{j=1}^{K} \exp(\text{top\_k\_logits}_j)}$$

This gives us $\mathbf{w} \in \mathbb{R}^{B \times S \times K}$ where:
- $w_i \geq 0$ for all $i$ (non-negative weights)
- $\sum_{i=1}^{K} w_i = 1$ for each token (weights sum to 1)

⚠️ **Pitfall — softmax scope matters!** Applying softmax over all $N$ experts *before* top-k selection gives different (worse) results. The correct order is: select top-k logits first, *then* softmax over just those $K$ values. This ensures the selected experts' weights always sum to exactly 1.0.

### Putting It All Together

The complete Top-K routing algorithm:

$$\mathbf{g} = \mathbf{x} \cdot \mathbf{W}_{\text{gate}} \quad \in \mathbb{R}^{B \times S \times N}$$

$$\text{top\_k\_logits}, \text{indices} = \text{TopK}(\mathbf{g}, k) \quad \in \mathbb{R}^{B \times S \times K}$$

$$\mathbf{w} = \text{softmax}(\text{top\_k\_logits}) \quad \in \mathbb{R}^{B \times S \times K}$$

The final MoE output for each token is:

$$\mathbf{y} = \sum_{i=1}^{K} w_i \cdot \text{Expert}_{\text{indices}_i}(\mathbf{x})$$

🎯 **Interview tip:** Be ready to explain *why* softmax is applied after top-k, not before. If you softmax first, the non-selected experts still "steal" probability mass, making the selected weights smaller and less decisive.

### Code: Top-K Routing Step by Step

Let's implement each step and verify the shapes and properties.

In [ ]:
# === Top-K Routing: Step-by-Step Implementation ===

# Configuration
d_model = 64       # Model dimension
num_experts = 8    # Total number of experts (N)
num_active = 2     # Experts active per token (K)
batch, seq = 2, 4  # Small batch for demonstration

# Create sample input
x = mx.random.normal((batch, seq, d_model))
print(f"Input x shape: {x.shape}")  # [2, 4, 64]

# Step 1: Gating logits — simple linear projection
W_gate = mx.random.normal((d_model, num_experts)) * 0.02  # Small init
router_logits = x @ W_gate
print(f"Router logits shape: {router_logits.shape}")  # [2, 4, 8]
assert router_logits.shape == (batch, seq, num_experts), "Shape mismatch!"

# Step 2: Top-K selection
# MLX doesn't have a direct top_k, so we use argpartition + gather
# We'll implement a clean top_k helper
def top_k(x, k):
    """Select top-k values and indices along the last axis."""
    # Get indices that would sort in descending order
    indices = mx.argpartition(-x, kth=k - 1, axis=-1)[..., :k]
    # Gather the corresponding values
    values = mx.take_along_axis(x, indices, axis=-1)
    return values, indices

top_k_logits, top_k_indices = top_k(router_logits, num_active)
print(f"Top-K logits shape: {top_k_logits.shape}")    # [2, 4, 2]
print(f"Top-K indices shape: {top_k_indices.shape}")   # [2, 4, 2]
assert top_k_logits.shape == (batch, seq, num_active)
assert top_k_indices.shape == (batch, seq, num_active)

# Step 3: Softmax over selected experts only
weights = mx.softmax(top_k_logits, axis=-1)
print(f"Weights shape: {weights.shape}")  # [2, 4, 2]
assert weights.shape == (batch, seq, num_active)

# Verify: weights sum to 1.0 for each token
weight_sums = mx.sum(weights, axis=-1)
print(f"\nWeight sums (should all be ~1.0):\n{weight_sums}")
assert mx.all(mx.abs(weight_sums - 1.0) < 1e-6), "Weights don't sum to 1!"

# Verify: all weights are non-negative
assert mx.all(weights >= 0), "Negative weights found!"

print("\n✅ All routing properties verified!")
print(f"\nExample — Token [0,0] routed to experts {top_k_indices[0, 0].tolist()}")
print(f"  with weights {weights[0, 0].tolist()}")

### Visualizing the Routing Decision

Let's see how the router distributes tokens across experts.

In [ ]:
# === Visualize Router Logits and Weight Distribution ===

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Plot 1: Raw router logits for first sequence
logits_np = np.array(router_logits[0])  # [seq, num_experts]
im = axes[0].imshow(logits_np, aspect='auto', cmap='RdBu_r')
axes[0].set_xlabel("Expert Index")
axes[0].set_ylabel("Token Position")
axes[0].set_title("Raw Router Logits (g = x @ W_gate)")
axes[0].set_xticks(range(num_experts))
plt.colorbar(im, ax=axes[0])

# Plot 2: After softmax — full distribution (for comparison)
full_probs = np.array(mx.softmax(router_logits[0], axis=-1))
im2 = axes[1].imshow(full_probs, aspect='auto', cmap='YlOrRd')
axes[1].set_xlabel("Expert Index")
axes[1].set_ylabel("Token Position")
axes[1].set_title("Full Softmax (all experts)")
axes[1].set_xticks(range(num_experts))
plt.colorbar(im2, ax=axes[1])

# Plot 3: Expert selection frequency across a larger batch
x_large = mx.random.normal((1, 128, d_model))
logits_large = x_large @ W_gate
_, indices_large = top_k(logits_large, num_active)
indices_flat = np.array(indices_large.reshape(-1))
counts = np.bincount(indices_flat, minlength=num_experts)
colors = ['#2ecc71' if c > 128 * num_active / num_experts * 0.5 else '#e74c3c' for c in counts]
axes[2].bar(range(num_experts), counts, color=colors)
axes[2].set_xlabel("Expert Index")
axes[2].set_ylabel("Tokens Routed")
axes[2].set_title("Expert Load (128 tokens, K=2)")
axes[2].axhline(y=128 * num_active / num_experts, color='black', linestyle='--',
                label=f'Ideal: {128 * num_active // num_experts}')
axes[2].legend()

plt.tight_layout()
plt.show()
print("💡 Green bars = well-utilized experts. Red bars = underutilized (risk of collapse).")

---

## MoE Configuration

Before building the full system, let's define the configuration that controls all MoE behavior. This dataclass will be used throughout the notebook.

🎯 **Interview tip:** Know these numbers cold — interviewers love asking "what are typical values for num_experts and num_active in Mixtral vs DeepSeek-V3?"

In [ ]:
# MoEConfig is imported from utils.moe — let's inspect it and show reference configs

# Reference configurations from real models
configs = {
    "Mixtral 8x7B":   MoEConfig(d_model=4096, num_experts=8,   num_active=2, d_ff=14336),
    "DeepSeek-V3":    MoEConfig(d_model=7168, num_experts=256,  num_active=8, d_ff=2048,
                                has_shared_expert=True),
    "Gemma 4 (27B)":  MoEConfig(d_model=4608, num_experts=128,  num_active=2, d_ff=768,
                                has_shared_expert=True),
    "Our notebook":   MoEConfig(),  # Small config for learning
}

print("Model Configurations:")
print(f"{'Model':<20} {'Experts':>8} {'Active':>7} {'d_model':>8} {'d_ff':>6} {'Shared':>7}")
print("-" * 60)
for name, cfg in configs.items():
    print(f"{name:<20} {cfg.num_experts:>8} {cfg.num_active:>7} {cfg.d_model:>8} "
          f"{cfg.d_ff:>6} {'✓' if cfg.has_shared_expert else '✗':>7}")

---

## Why Sparse > Dense at Scale

Let's make the MoE advantage concrete with numbers. The core question: **for a fixed compute budget, should you build one big dense model or a sparse MoE model?**

### Dense FFN Compute

In a standard transformer, the FFN has two linear layers:

$$\text{FFN}(\mathbf{x}) = \sigma(\mathbf{x} \mathbf{W}_1 + \mathbf{b}_1) \mathbf{W}_2 + \mathbf{b}_2$$

- $\mathbf{W}_1 \in \mathbb{R}^{d_{\text{model}} \times d_{\text{ff}}}$, $\mathbf{W}_2 \in \mathbb{R}^{d_{\text{ff}} \times d_{\text{model}}}$
- Parameters per FFN: $2 \times d_{\text{model}} \times d_{\text{ff}}$
- FLOPs per token: $\approx 2 \times 2 \times d_{\text{model}} \times d_{\text{ff}}$ (two matmuls)

### MoE FFN Compute

With $N$ experts and $K$ active per token:
- **Total parameters:** $N \times 2 \times d_{\text{model}} \times d_{\text{ff}}$ ($N\times$ more than dense)
- **Active parameters per token:** $K \times 2 \times d_{\text{model}} \times d_{\text{ff}}$ (only $K$ experts fire)
- **FLOPs per token:** $K/N$ of what a dense model with the same total params would cost

💡 **The MoE bargain:** You store $N$ experts in memory (more parameters = more knowledge), but each token only pays for $K$ of them (less compute = faster inference).

In [ ]:
# === Dense vs MoE: Parameter and Compute Comparison ===

def compute_ffn_stats(d_model, d_ff, num_experts=1, num_active=1, has_shared=False):
    """Compute parameter count and FLOPs for dense or MoE FFN."""
    params_per_expert = 2 * d_model * d_ff  # W1 + W2 (ignoring bias)
    total_params = num_experts * params_per_expert
    active_params = num_active * params_per_expert
    
    if has_shared:
        total_params += params_per_expert   # Shared expert adds one more
        active_params += params_per_expert  # Shared expert always active
    
    # Router params (small but let's count them)
    router_params = d_model * num_experts if num_experts > 1 else 0
    total_params += router_params
    
    # FLOPs ≈ 2 × params (one multiply + one add per parameter)
    flops_per_token = 2 * active_params
    
    return {
        "total_params": total_params,
        "active_params": active_params,
        "flops_per_token": flops_per_token,
        "sparsity": 1.0 - (active_params / (total_params - router_params)) if num_experts > 1 else 0.0,
    }

# Compare configurations
print(f"{'Model':<20} {'Total Params':>14} {'Active/Token':>14} {'FLOPs/Token':>14} {'Sparsity':>10}")
print("=" * 76)

comparisons = [
    ("Dense 7B-style",   dict(d_model=4096, d_ff=11008)),
    ("Dense 70B-style",  dict(d_model=8192, d_ff=28672)),
    ("Mixtral 8x7B",     dict(d_model=4096, d_ff=14336, num_experts=8, num_active=2)),
    ("DeepSeek-V3",      dict(d_model=7168, d_ff=2048, num_experts=256, num_active=8, has_shared=True)),
    ("Gemma 4 (27B)",    dict(d_model=4608, d_ff=768, num_experts=128, num_active=2, has_shared=True)),
]

for name, kwargs in comparisons:
    stats = compute_ffn_stats(**kwargs)
    print(f"{name:<20} {stats['total_params']:>14,} {stats['active_params']:>14,} "
          f"{stats['flops_per_token']:>14,} {stats['sparsity']:>9.1%}")

print("\n⚡ Notice: MoE models have MORE total params but FEWER active params per token.")

The next cell continues the implementation:

**=== Visualization: Dense vs MoE Scaling ===**

In [ ]:
# === Visualization: Dense vs MoE Scaling ===

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Total params vs Active params
d_model_ref = 4096
d_ff_ref = 14336
expert_counts = [1, 2, 4, 8, 16, 32, 64, 128]
k = 2  # Active experts

total_params = []
active_params = []
for n in expert_counts:
    stats = compute_ffn_stats(d_model_ref, d_ff_ref, num_experts=n, num_active=min(k, n))
    total_params.append(stats["total_params"] / 1e9)
    active_params.append(stats["active_params"] / 1e9)

axes[0].plot(expert_counts, total_params, 'o-', color='#e74c3c', linewidth=2, label='Total params (capacity)')
axes[0].plot(expert_counts, active_params, 's-', color='#2ecc71', linewidth=2, label='Active params/token (cost)')
axes[0].set_xlabel("Number of Experts (N)")
axes[0].set_ylabel("Parameters (Billions)")
axes[0].set_title("MoE Scaling: Capacity vs Cost (K=2)")
axes[0].set_xscale('log', base=2)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot 2: FLOPs ratio — MoE vs equivalent dense model
flops_ratios = [min(k, n) / n for n in expert_counts]
axes[1].bar(range(len(expert_counts)), flops_ratios, color='#3498db', alpha=0.8)
axes[1].set_xticks(range(len(expert_counts)))
axes[1].set_xticklabels([str(n) for n in expert_counts])
axes[1].set_xlabel("Number of Experts (N)")
axes[1].set_ylabel("FLOPs Ratio (MoE / Dense-equivalent)")
axes[1].set_title("Compute Savings with K=2 Active Experts")
axes[1].set_ylim(0, 1.1)
axes[1].axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='Dense baseline')
for i, r in enumerate(flops_ratios):
    axes[1].text(i, r + 0.03, f"{r:.0%}", ha='center', fontsize=9)
axes[1].legend()

plt.tight_layout()
plt.show()

print("💡 With 8 experts and K=2, you use only 25% of the compute for the same total capacity.")
print("   With 128 experts (Gemma 4), it drops to just 1.6%!")

---

## Summary So Far

We've covered the **why** and **math** of MoE:

| Concept | Key Formula | Shape |
|---------|------------|-------|
| Gating logits | $\mathbf{g} = \mathbf{x} \cdot \mathbf{W}_{\text{gate}}$ | $[B, S, N]$ |
| Top-K selection | $\text{TopK}(\mathbf{g}, k)$ | $[B, S, K]$ |
| Weight normalization | $\mathbf{w} = \text{softmax}(\text{top\_k\_logits})$ | $[B, S, K]$ |
| MoE output | $\mathbf{y} = \sum_{i=1}^{K} w_i \cdot \text{Expert}_i(\mathbf{x})$ | $[B, S, d_{\text{model}}]$ |

**Properties verified:**
- ✅ Weights sum to 1.0 (±1e-6) for every token
- ✅ All weights are non-negative
- ✅ Only K experts activated per token

🎯 **What's next:** We'll implement the full `MoERouter` module, `ExpertFFN`, and `MoEBlock` in MLX — including load balancing loss and shared experts.

---

## The MoERouter Module

Now let's wrap our step-by-step routing logic into a proper `nn.Module`. This is the reusable component that plugs into any MoE block.

The router has two jobs:
1. **`route(x)`** — given token embeddings, return which experts to use and how much to weight each
2. **`compute_load_balance_loss()`** — compute the auxiliary loss that prevents expert collapse

🎯 **Interview tip:** The router is the *only* learned component unique to MoE. Everything else (experts, residual connections) is standard transformer machinery. If the router fails, the whole MoE layer degrades to using just 1-2 experts.

In [ ]:
# MoERouter and top_k are imported from utils.moe
# Let's inspect the module structure and verify the router works

import inspect
print("=== MoERouter interface ===")
print(f"  route(x) -> (indices [B,S,K], weights [B,S,K])")
print(f"  compute_load_balance_loss() -> scalar")
print(f"  Learned gate: nn.Linear(d_model, num_experts, bias=False)")
print()

# Quick source peek at the route method signature
sig = inspect.signature(MoERouter.route)
print(f"MoERouter.route{sig}")
print()
print("✅ MoERouter imported from utils.moe — Top-K routing with load balance loss")

The next cell continues the implementation:

**=== Verify MoERouter ===**

In [ ]:
# === Verify MoERouter ===

config = MoEConfig()  # d_model=64, num_experts=8, num_active=2
router = MoERouter(config)

# Create test input
batch, seq = 2, 4
x = mx.random.normal((batch, seq, config.d_model))
print(f"Input shape: {x.shape}")

# Route tokens
indices, weights = router.route(x)
mx.eval(indices, weights)

print(f"Indices shape: {indices.shape}")   # [2, 4, 2]
print(f"Weights shape: {weights.shape}")   # [2, 4, 2]

# Verify shapes
assert indices.shape == (batch, seq, config.num_active), "❌ Indices shape wrong"
assert weights.shape == (batch, seq, config.num_active), "❌ Weights shape wrong"

# Verify weights sum to 1.0
weight_sums = mx.sum(weights, axis=-1)
mx.eval(weight_sums)
print(f"\nWeight sums (should be ~1.0):\n{weight_sums}")
assert mx.all(mx.abs(weight_sums - 1.0) < 1e-6).item(), "❌ Weights don't sum to 1.0!"

# Verify all weights non-negative
assert mx.all(weights >= 0).item(), "❌ Negative weights found!"

# Verify expert indices are valid
assert mx.all(indices >= 0).item(), "❌ Negative expert indices!"
assert mx.all(indices < config.num_experts).item(), "❌ Expert index out of range!"

# Compute load balance loss
aux_loss = router.compute_load_balance_loss()
mx.eval(aux_loss)
print(f"\nLoad balance loss: {aux_loss.item():.4f}")
print(f"  (Perfect balance = 1.0, higher = more imbalanced)")
assert aux_loss.item() >= 0, "❌ Negative load balance loss!"

print("\n✅ MoERouter: all shape, normalization, and loss checks passed!")

---

## Alternative Routing: Expert Choice

Top-K routing has a subtle problem: **popular experts get overloaded** while unpopular ones sit idle. The load balance loss helps, but it's a soft constraint.

**Expert Choice** (Zhou et al., 2022) flips the routing direction: instead of *tokens choosing experts*, **experts choose tokens**. Each expert selects its top-C tokens, where C is a fixed capacity per expert.

### How it works:

1. Compute router logits: $\mathbf{G} = \mathbf{X} \cdot \mathbf{W}_{\text{gate}} \in \mathbb{R}^{B \times S \times N}$
2. **Transpose** to get expert-centric view: $\mathbf{G}^T \in \mathbb{R}^{N \times (B \times S)}$
3. Each expert selects its top-C tokens: $C = \lfloor k \times B \times S / N \rfloor$
4. Apply softmax over selected tokens per expert

💡 **Key advantage:** Every expert processes *exactly* C tokens — perfect load balance by construction, no auxiliary loss needed.

⚠️ **Tradeoff:** Some tokens may be selected by fewer than k experts (or none!), while others may be selected by more. This means the effective compute per token varies.

In [ ]:
# ExpertChoiceRouter is imported from utils.moe
# Let's verify it works on a quick example

config = MoEConfig(d_model=64, num_experts=8, num_active=2)
ec_router = ExpertChoiceRouter(config)

x_test = mx.random.normal((2, 8, config.d_model))
indices, weights = ec_router.route(x_test)
mx.eval(indices, weights)

print(f"ExpertChoiceRouter — experts choose tokens, perfect load balance")
print(f"  Input:   {x_test.shape}")
print(f"  Indices: {indices.shape}  (which experts chose each token)")
print(f"  Weights: {weights.shape}  (normalised, sum to 1.0)")
print(f"  Sample [0,0]: experts {indices[0,0].tolist()}, weights {[f'{w:.3f}' for w in weights[0,0].tolist()]}")
print("✅ ExpertChoiceRouter imported from utils.moe")

---

## Alternative Routing: Hash-Based

The simplest possible routing: **no learned parameters at all**. Hash routing assigns tokens to experts using a deterministic hash function.

### How it works:

1. Hash each token's position (or content) to get a number
2. Map that number to expert indices: `expert = hash(position) % num_experts`
3. For k > 1, use k different hash functions (or offsets)
4. Weights are uniform: each selected expert gets weight 1/k

💡 **Why bother?** Hash routing is a useful *baseline*. If your learned router doesn't beat hash routing, something is wrong with your training. It's also used in some production systems where routing overhead matters.

⚡ **Performance:** Zero routing compute — no matrix multiply, no softmax. Just integer arithmetic. This matters at scale when the router itself becomes a bottleneck.

⚠️ **Limitation:** No input-dependent routing. A token always goes to the same expert regardless of content. This means hash routing can't specialize experts by topic or language.

In [ ]:
# HashRouter is imported from utils.moe
# Let's verify it works — deterministic, no learned parameters

config = MoEConfig(d_model=64, num_experts=8, num_active=2)
hash_router = HashRouter(config)

x_test = mx.random.normal((2, 8, config.d_model))
indices, weights = hash_router.route(x_test)
mx.eval(indices, weights)

print(f"HashRouter — zero-parameter deterministic routing")
print(f"  Input:   {x_test.shape}")
print(f"  Indices: {indices.shape}  (hash-assigned experts)")
print(f"  Weights: {weights.shape}  (uniform: each = 1/{config.num_active})")
print(f"  Sample [0,0]: experts {indices[0,0].tolist()}, weights {[f'{w:.3f}' for w in weights[0,0].tolist()]}")

# Verify determinism: same input → same routing
indices2, _ = hash_router.route(x_test)
mx.eval(indices2)
assert mx.array_equal(indices, indices2).item(), "Hash routing should be deterministic!"
print("  ✅ Deterministic: same input → same routing")
print("✅ HashRouter imported from utils.moe")

---

## Routing Methods: Side-by-Side Comparison

Let's run all three routing methods on the same input and compare their behavior. This is the key comparison for understanding the tradeoffs.

| Method | Learned? | Load Balance | Compute Cost | Used In |
|--------|----------|-------------|--------------|---------|
| **Top-K** | ✅ Yes | Soft (aux loss) | O(d × N) | Mixtral, DeepSeek-V3 |
| **Expert Choice** | ✅ Yes | Perfect (by design) | O(d × N + N × S) | V-MoE (Google) |
| **Hash** | ❌ No | Statistical | O(1) | Baselines, some production |

🎯 **Interview tip:** Be ready to explain *when* you'd pick each method. Top-K is the default. Expert Choice when load balance is critical. Hash when you need zero routing overhead or a baseline.

In [ ]:
# === Side-by-Side Comparison: All Three Routing Methods ===

config = MoEConfig(d_model=64, num_experts=8, num_active=2)

# Create all three routers
topk_router = MoERouter(config)
ec_router = ExpertChoiceRouter(config)
hash_router = HashRouter(config)

# Same input for all three
batch, seq = 2, 16
x = mx.random.normal((batch, seq, config.d_model))
print(f"Input shape: {x.shape}")
print(f"Config: {config.num_experts} experts, {config.num_active} active per token")
print("=" * 70)

# Route with each method
results = {}
for name, router in [("Top-K", topk_router), ("Expert Choice", ec_router), ("Hash", hash_router)]:
    indices, weights = router.route(x)
    mx.eval(indices, weights)
    
    # Verify shapes
    assert indices.shape == (batch, seq, config.num_active), \
        f"{name}: indices shape {indices.shape} != expected"
    assert weights.shape == (batch, seq, config.num_active), \
        f"{name}: weights shape {weights.shape} != expected"
    
    # Verify weight normalization
    weight_sums = mx.sum(weights, axis=-1)
    mx.eval(weight_sums)
    max_deviation = mx.max(mx.abs(weight_sums - 1.0)).item()
    
    # Verify non-negative weights
    min_weight = mx.min(weights).item()
    
    # Expert load distribution
    flat_indices = indices.reshape(-1).tolist()
    expert_counts = [0] * config.num_experts
    for idx in flat_indices:
        expert_counts[int(idx)] += 1
    
    total_assignments = len(flat_indices)
    ideal_per_expert = total_assignments / config.num_experts
    load_imbalance = max(expert_counts) / max(ideal_per_expert, 1)
    
    results[name] = {
        "indices": indices,
        "weights": weights,
        "max_weight_deviation": max_deviation,
        "min_weight": min_weight,
        "expert_counts": expert_counts,
        "load_imbalance": load_imbalance,
    }
    
    print(f"\n{'─' * 35}")
    print(f"🔀 {name} Router")
    print(f"{'─' * 35}")
    print(f"  Weight sum deviation: {max_deviation:.2e} (should be < 1e-6)")
    print(f"  Min weight: {min_weight:.4f} (should be ≥ 0)")
    print(f"  Expert load: {expert_counts}")
    print(f"  Load imbalance: {load_imbalance:.2f}x (1.0 = perfect)")
    print(f"  Sample routing [0,0]: experts {indices[0, 0].tolist()}, "
          f"weights {[f'{w:.3f}' for w in weights[0, 0].tolist()]}")

print(f"\n{'=' * 70}")
print("✅ All three routers produce valid (indices, weights) with correct shapes!")

The next cell continues the implementation:

**=== Visualization: Routing Comparison ===**

In [ ]:
# === Visualization: Routing Comparison ===

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

router_names = ["Top-K", "Expert Choice", "Hash"]
colors_map = {"Top-K": "#3498db", "Expert Choice": "#2ecc71", "Hash": "#e67e22"}

for ax, name in zip(axes, router_names):
    counts = results[name]["expert_counts"]
    ideal = sum(counts) / config.num_experts
    
    bar_colors = []
    for c in counts:
        if c == 0:
            bar_colors.append("#e74c3c")  # Red: unused expert
        elif abs(c - ideal) / max(ideal, 1) < 0.3:
            bar_colors.append("#2ecc71")  # Green: well-balanced
        else:
            bar_colors.append("#f39c12")  # Orange: somewhat imbalanced
    
    ax.bar(range(config.num_experts), counts, color=bar_colors, alpha=0.85, edgecolor="white")
    ax.axhline(y=ideal, color="black", linestyle="--", alpha=0.5, label=f"Ideal: {ideal:.0f}")
    ax.set_xlabel("Expert Index")
    ax.set_ylabel("Tokens Assigned")
    ax.set_title(f"{name} Router\n(imbalance: {results[name]['load_imbalance']:.2f}x)")
    ax.set_xticks(range(config.num_experts))
    ax.legend(fontsize=9)
    ax.set_ylim(0, max(max(counts) + 2, ideal * 2))

plt.suptitle(f"Expert Load Distribution — {batch}×{seq} tokens, K={config.num_active}", 
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

print("💡 Green = well-utilized, Orange = somewhat imbalanced, Red = unused (expert collapse risk)")

The next cell continues the implementation:

**=== Comprehensive Verification: Weight Normalization & Shape Properties ===**

In [ ]:
# === Comprehensive Verification: Weight Normalization & Shape Properties ===

print("=" * 70)
print("🔍 VERIFICATION: Weight Normalization & Shape Properties")
print("=" * 70)

# Test with multiple input sizes to ensure robustness
test_configs = [
    (1, 1, "Single token"),
    (1, 8, "Single batch, 8 tokens"),
    (4, 16, "4 batches, 16 tokens"),
    (2, 32, "2 batches, 32 tokens"),
]

config_v = MoEConfig(d_model=64, num_experts=8, num_active=2)
all_passed = True

for batch_test, seq_test, desc in test_configs:
    x_test = mx.random.normal((batch_test, seq_test, config_v.d_model))
    
    routers = [
        ("Top-K",          MoERouter(config_v)),
        ("Expert Choice",  ExpertChoiceRouter(config_v)),
        ("Hash",           HashRouter(config_v)),
    ]
    
    for name, router in routers:
        indices, weights = router.route(x_test)
        mx.eval(indices, weights)
        
        # Property 1: Shape correctness
        shape_ok = (indices.shape == (batch_test, seq_test, config_v.num_active) and
                    weights.shape == (batch_test, seq_test, config_v.num_active))
        
        # Property 2: Weights sum to 1.0 (±1e-5 tolerance for float32 accumulation)
        weight_sums = mx.sum(weights, axis=-1)
        mx.eval(weight_sums)
        sum_ok = mx.all(mx.abs(weight_sums - 1.0) < 1e-5).item()
        
        # Property 3: Non-negative weights
        nonneg_ok = mx.all(weights >= 0).item()
        
        # Property 4: Valid expert indices
        valid_idx_ok = (mx.all(indices >= 0).item() and 
                        mx.all(indices < config_v.num_experts).item())
        
        # Property 5: Exactly num_active experts per token
        count_ok = indices.shape[-1] == config_v.num_active
        
        passed = shape_ok and sum_ok and nonneg_ok and valid_idx_ok and count_ok
        
        if not passed:
            all_passed = False
            print(f"  ❌ {name:15s} | {desc:25s} | "
                  f"shape={shape_ok} sum={sum_ok} nonneg={nonneg_ok} "
                  f"valid_idx={valid_idx_ok} count={count_ok}")
    
    print(f"  ✅ {desc:25s} — all 3 routers passed all properties")

print(f"\n{'=' * 70}")
if all_passed:
    print("✅ ALL VERIFICATION CHECKS PASSED!")
    print("   • Shapes: [batch, seq, k] for both indices and weights")
    print("   • Weight normalization: sum = 1.0 (±1e-5) for every token")
    print("   • Non-negative weights: all w ≥ 0")
    print("   • Valid expert indices: 0 ≤ idx < num_experts")
    print("   • Expert count: exactly num_active experts per token")
else:
    print("❌ SOME CHECKS FAILED — see details above")

---

## Expert FFN: The Building Block

Each expert is a standard feed-forward network. Modern MoE models (Mixtral, Gemma 4, DeepSeek-V3) use **SwiGLU** activation — a gated variant of SiLU that's become the default in 2024-2025 LLMs.

### SwiGLU Architecture

Three weight matrices instead of two:

$\text{ExpertFFN}(\mathbf{x}) = \mathbf{W}_2 \big( \text{SiLU}(\mathbf{W}_1 \mathbf{x}) \odot \mathbf{W}_3 \mathbf{x} \big)$

Where:
- $\mathbf{W}_1$ (gate_proj): $d_{\text{model}} \to d_{\text{ff}}$ — controls the gate
- $\mathbf{W}_3$ (up_proj): $d_{\text{model}} \to d_{\text{ff}}$ — projects up
- $\mathbf{W}_2$ (down_proj): $d_{\text{ff}} \to d_{\text{model}}$ — projects back down
- $\odot$ is element-wise multiplication

💡 **Why SwiGLU?** The gating mechanism lets the network learn *which* features to pass through, giving better gradient flow than plain ReLU or GELU. It's ~1% more parameters but consistently better loss.

🎯 **Interview tip:** Know that SwiGLU has 3 matrices (not 2). This means each expert has $3 \times d_{\text{model}} \times d_{\text{ff}}$ parameters, not $2 \times d_{\text{model}} \times d_{\text{ff}}$.

In [ ]:
# === Shared Expert Demo ===

# Without shared expert
config_no_shared = MoEConfig(d_model=64, num_experts=8, num_active=2, d_ff=256,
                            has_shared_expert=False)
moe_no_shared = MoEBlock(config_no_shared)

# With shared expert
config_shared = MoEConfig(d_model=64, num_experts=8, num_active=2, d_ff=256,
                          has_shared_expert=True)
moe_shared = MoEBlock(config_shared)

x = mx.random.normal((2, 8, 64))

out_no, loss_no = moe_no_shared(x)
out_shared, loss_shared = moe_shared(x)
mx.eval(out_no, loss_no, out_shared, loss_shared)

print("Shared Expert Comparison:")
print(f"  {'':20s} {'No Shared':>12s} {'With Shared':>12s}")
print(f"  {'Output shape':20s} {str(out_no.shape):>12s} {str(out_shared.shape):>12s}")
print(f"  {'Aux loss':20s} {loss_no.item():>12.6f} {loss_shared.item():>12.6f}")
print(f"  {'Has shared expert':20s} {'No':>12s} {'Yes':>12s}")
print()

# Both must preserve shape
assert out_no.shape == x.shape, "Shape mismatch without shared expert!"
assert out_shared.shape == x.shape, "Shape mismatch with shared expert!"

# Count experts: routed + shared
routed_experts = config_shared.num_experts
shared_count = 1 if config_shared.has_shared_expert else 0
total_experts = routed_experts + shared_count
active_per_token = config_shared.num_active + shared_count

print(f"  Total expert FFNs:     {total_experts} ({routed_experts} routed + {shared_count} shared)")
print(f"  Active per token:      {active_per_token} ({config_shared.num_active} routed + {shared_count} shared)")
print(f"  Compute ratio:         {active_per_token}/{total_experts} = {active_per_token/total_experts:.0%}")
print()
print("✅ Shared expert: output shape preserved, always-active expert adds to routed output")

The next cell continues the implementation:

**=== ExpertFFN: SwiGLU Feed-Forward Network ===**

In [ ]:
# === ExpertFFN: SwiGLU Feed-Forward Network ===
# Imported from utils.moe — let's verify it works

config = MoEConfig(d_model=64, num_experts=8, num_active=2, d_ff=256)
expert = ExpertFFN(config)

# Test with a small input
batch, seq = 2, 4
x = mx.random.normal((batch, seq, config.d_model))
out = expert(x)
mx.eval(out)

print(f"ExpertFFN forward pass:")
print(f"  Input shape:  {x.shape}")
print(f"  Output shape: {out.shape}")
assert out.shape == x.shape, f"Shape mismatch: {out.shape} != {x.shape}"

# Count parameters per expert
params = sum(p.size for _, p in expert.parameters().items() if hasattr(p, 'size'))
# Manual count: gate_proj + up_proj + down_proj
expected = config.d_model * config.d_ff * 3  # 3 matrices for SwiGLU
print(f"  Parameters:   {expected:,} (3 × {config.d_model} × {config.d_ff})")
print(f"  Architecture: W2( SiLU(W1·x) ⊙ W3·x )")

# Verify no NaN/Inf
assert mx.all(mx.isfinite(out)).item(), "NaN/Inf in expert output!"
print("\n✅ ExpertFFN: shape preserved, no NaN/Inf, SwiGLU activation working")

---

## Shared Expert: Gemma 4 / DeepSeek-V3 Style

Modern MoE architectures add a **shared expert** that processes ALL tokens, regardless of routing. The shared expert captures common patterns that every token needs, while the routed experts specialize.

$\mathbf{y} = \underbrace{\sum_{i=1}^{K} w_i \cdot \text{Expert}_i(\mathbf{x})}_{\text{routed experts}} + \underbrace{\text{SharedExpert}(\mathbf{x})}_{\text{always active}}$

💡 **Why shared experts help:** Without a shared expert, common knowledge (like grammar, basic facts) must be duplicated across all experts. The shared expert handles the "common" work, freeing routed experts to truly specialize.

🎯 **Interview tip:** Gemma 4 uses 128 routed experts + 1 shared expert with K=2. DeepSeek-V3 uses 256 routed experts + 1 shared expert with K=8. The shared expert is the same architecture as a routed expert — just always active.

---

## The MoE Block: Putting It All Together

The `MoEBlock` is the complete MoE layer that replaces the standard FFN in a transformer block. It combines:

1. **Router** — decides which experts handle each token
2. **Expert FFNs** — the actual computation (one per expert)
3. **Shared Expert** (optional) — processes ALL tokens (Gemma 4 / DeepSeek-V3 style)
4. **Load Balance Loss** — prevents expert collapse

### Forward Pass Algorithm

```
for each token:
    1. Router selects top-k experts and weights
    2. Each selected expert processes the token
    3. Outputs are combined: y = Σ(w_i × Expert_i(x))
    4. If shared expert: y += SharedExpert(x)
    5. aux_loss = load_balance_weight × router.load_balance_loss()
```

⚠️ **OOM Recovery:** If the forward pass exceeds the 20GB memory budget, the block automatically falls back to sequential expert evaluation (one expert at a time, freeing memory between experts).

🎯 **Interview tip:** The output shape MUST equal the input shape `[batch, seq, d_model]`. MoE is a drop-in replacement for the FFN — the rest of the transformer doesn't know it's there.

In [ ]:
# === Load Balance Loss Demo ===

config = MoEConfig(d_model=64, num_experts=8, num_active=2, d_ff=256,
                   load_balance_weight=0.01)
moe = MoEBlock(config)

# Run forward pass to populate router state
x = mx.random.normal((4, 16, config.d_model))
output, aux_loss = moe(x)
mx.eval(output, aux_loss)

# Get the raw load balance loss (before weighting)
raw_loss = moe.router.compute_load_balance_loss()
mx.eval(raw_loss)

print("Load Balance Loss Breakdown:")
print(f"  Raw loss (num_experts × Σ(f_i × p_i)): {raw_loss.item():.4f}")
print(f"  Weighted loss (raw × {config.load_balance_weight}):  {aux_loss.item():.6f}")
print(f"  Perfect balance would give raw loss ≈ {config.num_active:.1f}")
print()

# Verify the formula: aux_loss = load_balance_weight * raw_loss
expected_aux = config.load_balance_weight * raw_loss.item()
print(f"  Verification: {config.load_balance_weight} × {raw_loss.item():.4f} = {expected_aux:.6f}")
print(f"  Actual aux_loss: {aux_loss.item():.6f}")
assert abs(aux_loss.item() - expected_aux) < 1e-5, "Loss formula mismatch!"
print("\n✅ Load balance loss formula verified: load_balance_weight × num_experts × Σ(f_i × p_i)")

The next cell continues the implementation:

**=== MoEBlock: Full MoE Layer ===**

In [ ]:
# === MoEBlock: Full MoE Layer ===
# Imported from utils.moe — let's run a forward pass and verify everything

config = MoEConfig(d_model=64, num_experts=8, num_active=2, d_ff=256)
moe_block = MoEBlock(config)

batch, seq = 2, 4
x = mx.random.normal((batch, seq, config.d_model))

# Forward pass returns (output, aux_loss)
output, aux_loss = moe_block(x)
mx.eval(output, aux_loss)

print("MoEBlock forward pass:")
print(f"  Input shape:  {x.shape}")
print(f"  Output shape: {output.shape}")
print(f"  Aux loss:     {aux_loss.item():.6f}")
print()

# Critical assertion: output shape == input shape
assert output.shape == x.shape, (
    f"❌ Shape mismatch: output {output.shape} != input {x.shape}"
)
print("✅ Shape preservation: output shape == input shape")

# Verify aux_loss is non-negative
assert aux_loss.item() >= 0, f"❌ Negative aux loss: {aux_loss.item()}"
print(f"✅ Load balance loss is non-negative: {aux_loss.item():.6f}")

# Verify no NaN/Inf
assert mx.all(mx.isfinite(output)).item(), "❌ NaN/Inf in output!"
print("✅ No NaN/Inf in output")

print(f"\n⚡ Config: {config.num_experts} experts, {config.num_active} active per token")
print(f"   Each token uses {config.num_active}/{config.num_experts} = {config.num_active/config.num_experts:.0%} of expert compute")

---

## Load Balancing Loss: Preventing Expert Collapse

Without load balancing, the router tends to send most tokens to just 1-2 "favorite" experts. The others never get trained and become useless — this is **expert collapse**.

The auxiliary loss encourages uniform expert utilization:

$\mathcal{L}_{\text{aux}} = N \times \sum_{i=1}^{N} f_i \cdot p_i$

Where:
- $N$ = number of experts
- $f_i$ = fraction of tokens actually routed to expert $i$
- $p_i$ = mean routing probability for expert $i$ (from softmax over all experts)

💡 **Why this formula works:** When load is perfectly balanced, $f_i = K/N$ and $p_i = 1/N$ for all experts, giving $\mathcal{L}_{\text{aux}} = K \approx 1.0$. When load is imbalanced, the product $f_i \cdot p_i$ is larger for overloaded experts, pushing the loss above 1.0.

⚠️ **The weight matters:** `load_balance_weight` (typically 0.01) controls how much the auxiliary loss influences training. Too low → expert collapse. Too high → routing becomes uniform (ignoring token content).

---

## 📊 Memory Analysis: MoE vs Dense Models

The MoE tradeoff is fundamentally about **memory vs compute**. You need to *store* all expert weights in memory (more total parameters), but each token only *activates* a fraction of them (less compute per token).

This has real consequences for deployment:

- 💡 **Total params** determine how much RAM/VRAM you need to load the model
- ⚡ **Active params per token** determine inference speed (FLOPs per token)
- ⚠️ **Memory footprint** depends on precision: float32 (4 bytes), bfloat16 (2 bytes), int8 (1 byte), int4 (0.5 bytes)

🎯 **Interview tip:** When someone says "Mixtral 8x7B is a 47B model that runs like a 13B model," they mean: 47B total params (memory cost) but only ~13B active per token (compute cost). You need the RAM for 47B but get the speed of 13B.

Let's compute exact numbers for real models.

In [ ]:
# === Memory Analysis: Real Model Comparisons ===
# Using published parameter counts for real-world models

models = {
    "LLaMA 7B":     {"total_params_B": 6.7,  "active_params_B": 6.7,  "type": "Dense",
                      "num_experts": 1, "num_active": 1, "has_shared": False},
    "LLaMA 70B":    {"total_params_B": 70.0,  "active_params_B": 70.0,  "type": "Dense",
                      "num_experts": 1, "num_active": 1, "has_shared": False},
    "Mixtral 8x7B": {"total_params_B": 46.7,  "active_params_B": 12.9,  "type": "MoE",
                      "num_experts": 8, "num_active": 2, "has_shared": False},
    "DeepSeek-V3":  {"total_params_B": 671.0, "active_params_B": 36.7,  "type": "MoE",
                      "num_experts": 256, "num_active": 8, "has_shared": True},
    "Gemma 4 27B":  {"total_params_B": 27.0,  "active_params_B": 4.9,   "type": "MoE",
                      "num_experts": 128, "num_active": 2, "has_shared": True},
}

# Compute memory footprint at different precisions
bytes_per_param = {"float32": 4, "bfloat16": 2, "int8": 1, "int4": 0.5}

print("=" * 100)
print("📊 MEMORY ANALYSIS: MoE vs Dense Models")
print("=" * 100)
print()

# Detailed table
header = f"{'Model':<16} {'Type':<6} {'Experts':>8} {'Active':>7} {'Total':>8} {'Active':>8} {'Sparsity':>9}"
header += f" {'FP32':>8} {'BF16':>8} {'INT4':>8}"
print(header)
print(f"{'':16} {'':6} {'(N)':>8} {'(K)':>7} {'Params':>8} {'/Token':>8} {'':>9} {'(GB)':>8} {'(GB)':>8} {'(GB)':>8}")
print("-" * 100)

for name, info in models.items():
    total_B = info["total_params_B"]
    active_B = info["active_params_B"]
    sparsity = 1.0 - (active_B / total_B) if total_B > 0 else 0.0
    
    mem_fp32 = total_B * bytes_per_param["float32"]
    mem_bf16 = total_B * bytes_per_param["bfloat16"]
    mem_int4 = total_B * bytes_per_param["int4"]
    
    print(f"{name:<16} {info['type']:<6} {info['num_experts']:>8} {info['num_active']:>7} "
          f"{total_B:>7.1f}B {active_B:>7.1f}B {sparsity:>8.1%} "
          f"{mem_fp32:>7.1f}G {mem_bf16:>7.1f}G {mem_int4:>7.1f}G")

print("-" * 100)
print()

# Highlight the key insight
print("💡 Key Insights:")
print(f"   • Mixtral 8x7B: 47B params but only 13B active → runs like a 13B model, needs RAM for 47B")
print(f"   • DeepSeek-V3:  671B params but only 37B active → 94.5% of experts idle per token!")
print(f"   • Gemma 4 27B:  27B params but only 5B active  → fits on consumer hardware (14GB in INT4)")
print()
print("⚠️ The memory wall: you must load ALL params into memory, even though most are idle per token.")
print("   This is why quantization (INT4/INT8) is critical for MoE deployment.")

The next cell continues the implementation:

**=== Visualization: MoE vs Dense — Total Params, Active Params, Memory ===**

In [ ]:
# === Visualization: MoE vs Dense — Total Params, Active Params, Memory ===

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

model_names = list(models.keys())
total_params = [models[m]["total_params_B"] for m in model_names]
active_params = [models[m]["active_params_B"] for m in model_names]
model_types = [models[m]["type"] for m in model_names]

# Color by type: blue for dense, green for MoE
colors_total = ["#3498db" if t == "Dense" else "#e74c3c" for t in model_types]
colors_active = ["#85c1e9" if t == "Dense" else "#2ecc71" for t in model_types]

x_pos = np.arange(len(model_names))
bar_width = 0.35

# --- Plot 1: Total vs Active Parameters (grouped bar) ---
bars1 = axes[0].bar(x_pos - bar_width/2, total_params, bar_width,
                     label="Total Params", color=colors_total, edgecolor="white", alpha=0.9)
bars2 = axes[0].bar(x_pos + bar_width/2, active_params, bar_width,
                     label="Active / Token", color=colors_active, edgecolor="white", alpha=0.9)

# Add value labels on bars (skip DeepSeek-V3 total for readability)
for i, (tp, ap) in enumerate(zip(total_params, active_params)):
    axes[0].text(i - bar_width/2, tp + 5, f"{tp:.0f}B", ha="center", fontsize=7, fontweight="bold")
    axes[0].text(i + bar_width/2, ap + 5, f"{ap:.0f}B", ha="center", fontsize=7, fontweight="bold")

axes[0].set_ylabel("Parameters (Billions)")
axes[0].set_title("Total vs Active Parameters per Token")
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(model_names, rotation=25, ha="right", fontsize=9)
axes[0].legend(fontsize=9)
axes[0].grid(axis="y", alpha=0.3)

# --- Plot 2: Memory Footprint at Different Precisions (stacked bar) ---
mem_bf16 = [models[m]["total_params_B"] * 2 for m in model_names]
mem_int8 = [models[m]["total_params_B"] * 1 for m in model_names]
mem_int4 = [models[m]["total_params_B"] * 0.5 for m in model_names]

# Show as grouped bars for each precision
bw = 0.25
axes[1].bar(x_pos - bw, mem_bf16, bw, label="BFloat16 (2B/param)", color="#9b59b6", alpha=0.85)
axes[1].bar(x_pos,      mem_int8, bw, label="INT8 (1B/param)",     color="#3498db", alpha=0.85)
axes[1].bar(x_pos + bw, mem_int4, bw, label="INT4 (0.5B/param)",   color="#2ecc71", alpha=0.85)

# Reference lines for common hardware
axes[1].axhline(y=48, color="#e74c3c", linestyle="--", alpha=0.6, label="M4 Pro 48GB")
axes[1].axhline(y=24, color="#f39c12", linestyle="--", alpha=0.6, label="RTX 4090 24GB")
axes[1].axhline(y=16, color="#95a5a6", linestyle="--", alpha=0.4, label="16GB GPU")

axes[1].set_ylabel("Memory (GB)")
axes[1].set_title("Memory Footprint by Precision")
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(model_names, rotation=25, ha="right", fontsize=9)
axes[1].legend(fontsize=7, loc="upper left")
axes[1].grid(axis="y", alpha=0.3)
axes[1].set_ylim(0, max(mem_bf16) * 1.15)

# --- Plot 3: Efficiency Ratio (Active/Total) ---
efficiency = [a / t * 100 for a, t in zip(active_params, total_params)]
bar_colors_eff = ["#3498db" if t == "Dense" else "#2ecc71" for t in model_types]

bars_eff = axes[2].bar(x_pos, efficiency, 0.6, color=bar_colors_eff, edgecolor="white", alpha=0.9)

for i, (eff, name) in enumerate(zip(efficiency, model_names)):
    axes[2].text(i, eff + 1.5, f"{eff:.1f}%", ha="center", fontsize=9, fontweight="bold")

axes[2].set_ylabel("Active / Total Params (%)")
axes[2].set_title("Compute Efficiency (lower = more sparse)")
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels(model_names, rotation=25, ha="right", fontsize=9)
axes[2].set_ylim(0, 115)
axes[2].axhline(y=100, color="gray", linestyle=":", alpha=0.4)
axes[2].grid(axis="y", alpha=0.3)

# Add legend for model types
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor="#3498db", label="Dense"),
                   Patch(facecolor="#2ecc71", label="MoE")]
axes[2].legend(handles=legend_elements, fontsize=9)

plt.suptitle("MoE vs Dense Models: The Memory-Compute Tradeoff", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

print("⚡ MoE models trade memory (must store all experts) for compute (only activate a few).")
print("   Quantization (INT4) makes even 671B DeepSeek-V3 feasible on high-end hardware.")

### 🎯 Memory Analysis Takeaways

| Metric | Dense Models | MoE Models |
|--------|-------------|------------|
| **Total params** | = Active params | Much larger (N× experts) |
| **Active params/token** | = Total params | K/N fraction of total |
| **Memory to load** | Proportional to quality | Disproportionately large |
| **Inference speed** | Proportional to size | Much faster than size suggests |
| **Quantization impact** | Helpful | **Critical** — makes deployment feasible |

💡 **The MoE deployment equation:**
- You need enough RAM to hold **all** expert weights (total params × bytes/param)
- But inference speed scales with **active** params, not total params
- INT4 quantization cuts memory 8× vs float32 — turning a 671B model from "needs a server" to "fits on high-end workstation"

⚠️ **Hidden cost:** MoE models also need memory for the router's gating computation and for shuffling tokens between experts. At scale (256 experts), this routing overhead is ~1-2% of total compute — small but not zero.

⚡ **Why Apple Silicon helps:** Unified memory means CPU and GPU share the same RAM pool. A 48GB M4 Pro can load models that would need separate CPU RAM + GPU VRAM on discrete GPU systems. This is especially valuable for MoE models where total params are large but active compute is small.

---

## 📊 Deep Dive: MoE Memory Overhead Analysis

The cells above used hardcoded model stats. Now let's use `MoEConfig` directly to compute **exact** parameter counts from architecture configs — and visualize the *memory overhead ratio*: how much extra memory MoE requires relative to what it actually uses per token.

💡 **Why this matters:** A dense model's memory is 100% "useful" — every stored parameter is active for every token. An MoE model stores N× more parameters but only uses K of them. The overhead ratio tells you how much you're "paying" in memory for capacity you don't use on any single token.

⚡ **Deployment rule of thumb:** If overhead ratio is 4×, you need 4× the RAM of an equivalent-quality dense model, but get 4× the capacity (knowledge) at the same inference speed.

In [ ]:
# === MoE Memory Overhead Analysis (from MoEConfig) ===
from utils.moe_analysis import compute_moe_vs_dense_stats, plot_moe_vs_dense_params

# Define model configs using MoEConfig
model_configs = [
    {"name": "Dense 7B",     "type": "Dense", "config": MoEConfig(d_model=4096, d_ff=11008, num_experts=1, num_active=1)},
    {"name": "Dense 70B",    "type": "Dense", "config": MoEConfig(d_model=8192, d_ff=28672, num_experts=1, num_active=1)},
    {"name": "Mixtral 8x7B", "type": "MoE",   "config": MoEConfig(d_model=4096, d_ff=14336, num_experts=8, num_active=2)},
    {"name": "DeepSeek-V3",  "type": "MoE",   "config": MoEConfig(d_model=7168, d_ff=2048, num_experts=256, num_active=8, has_shared_expert=True)},
    {"name": "Gemma 4 27B",  "type": "MoE",   "config": MoEConfig(d_model=4608, d_ff=768, num_experts=128, num_active=2, has_shared_expert=True)},
]

stats = compute_moe_vs_dense_stats(model_configs)

# Print overhead analysis
print("Memory Overhead Analysis (FFN layer only, SwiGLU 3-matrix experts):")
print(f"{'Model':<16} {'Total':>10} {'Active':>10} {'Overhead':>10} {'Sparsity':>10} {'BF16 Mem':>10}")
print("-" * 68)
for s in stats:
    print(f"{s['name']:<16} {s['total_params']/1e9:>9.2f}B {s['active_params']/1e9:>9.2f}B "
          f"{s['overhead_ratio']:>9.1f}x {s['sparsity']:>9.1%} {s['memory_gb']['bfloat16']:>9.2f}G")

print("\n🎯 Overhead ratio = how many times more memory you need vs what's active per token.")

The next cell continues the implementation:

**=== Visualization: Total vs Active Params + Memory Overhead ===**

In [ ]:
# === Visualization: Total vs Active Params + Memory Overhead ===
fig = plot_moe_vs_dense_params(stats)
plt.show()

print("⚠️ Higher overhead ratio = more 'idle' memory. DeepSeek-V3 stores ~18× more params than it uses per token!")

### ⚡ MoE Scaling Curve: Capacity vs Cost as Experts Grow

What happens as you add more experts? Total capacity (parameters) grows linearly with N, but active cost stays flat at K experts. The shaded region shows the growing gap — **idle parameters** that consume memory but don't contribute to any single token's computation.

🎯 **Interview insight:** This is the fundamental MoE scaling law. Doubling experts doubles capacity (and memory) but keeps per-token cost constant. It's "free" capacity — if you have the RAM.

In [ ]:
# === MoE Scaling Curve: Capacity vs Cost ===
from utils.moe_analysis import plot_moe_scaling_curve

fig = plot_moe_scaling_curve(
    d_model=4096,
    d_ff=14336,
    expert_counts=[1, 2, 4, 8, 16, 32, 64, 128, 256],
    k=2,
)
plt.show()

print("💡 At 256 experts (DeepSeek-V3 scale), overhead is 128× and sparsity is 99.2%.")
print("   You need 128× the memory of a dense model with the same per-token compute.")

---

## 📜 History & Alternatives

The idea of routing different inputs to specialized sub-networks is older than transformers themselves — it dates back to 1991. What changed is *scale*: modern MoE systems route trillions of tokens across hundreds of experts on thousands of accelerators.

Here's the full timeline from dense networks to today's sparse giants.

### Chronological Timeline

| Year | Model / Paper | Team | Key Contribution |
|------|--------------|------|-----------------|
| **1991** | Mixture of Experts | Jacobs, Jordan, Nowlan, Hinton | 💡 Original MoE concept — a gating network learns to partition the input space among specialist networks. Foundational idea that every modern MoE builds on. |
| **2017** | Dense FFN (Transformer) | Vaswani et al., Google | The "Attention Is All You Need" transformer uses a single dense FFN per layer. Every token activates every parameter — simple but expensive at scale. |
| **2017** | Sparsely-Gated MoE | Shazeer et al., Google | ⚡ First large-scale MoE for NLP. Scaled to 137B parameters with thousands of experts. Introduced the noisy top-k gating mechanism and load balancing loss. Proved MoE could work at scale, but training was notoriously unstable. |
| **2021** | Switch Transformer | Fedus, Zoph, Dean, Google | 💡 Simplified routing to **K=1** (each token goes to exactly one expert). Scaled to **1.6 trillion parameters** — the largest model at the time. Key insight: simpler routing + better load balancing = more stable training. |
| **2021** | V-MoE (Vision MoE) | Riquelme et al., Google | Introduced **Expert Choice routing** — experts choose their tokens instead of tokens choosing experts. Guarantees perfect load balance by construction. Applied MoE to Vision Transformers. |
| **2022** | ST-MoE | Zoph et al., Google | ⚡ Focused on **stable training techniques** for MoE at scale. Introduced router z-loss, improved auxiliary losses, and systematic study of MoE training instabilities. Made MoE practical for production. |
| **2023** | Mixtral 8×7B | Mistral AI | 🎯 First **open-weight MoE LLM**. N=8 experts, K=2 active per token. 47B total params but only ~13B active — matched or beat LLaMA 2 70B at a fraction of the inference cost. Made MoE accessible to the open-source community. |
| **2025** | DeepSeek-V3 | DeepSeek | ⚡ Massive scale: **671B total params**, N=256 experts, K=8 active, plus a **shared expert** that processes all tokens. Introduced **auxiliary-loss-free routing** using bias terms instead of load balance loss. Trained on 14.8T tokens. |
| **2026** | Gemma 4 | Google DeepMind | 💡 **128 routed experts + 1 shared expert**, K=2 active per token. Combines the shared expert pattern (DeepSeek-V3 style) with efficient routing at Google scale. Demonstrates that MoE + shared expert is the converging design pattern for frontier models. |

### 💡 Key Trends in the Timeline

Three clear patterns emerge from this evolution:

**1. Routing got simpler, not more complex**
- 1991: Complex gating networks with multiple layers
- 2017: Noisy top-k with learned noise
- 2021: Switch Transformer dropped to K=1 — *simpler is better*
- 2025: DeepSeek-V3 removed auxiliary loss entirely, using bias terms instead

**2. The shared expert pattern converged**
- Early MoE: all experts are routed (no shared component)
- 2025: DeepSeek-V3 adds a shared expert that sees every token
- 2026: Gemma 4 adopts the same pattern (128 routed + 1 shared)
- ⚡ The shared expert captures common patterns, freeing routed experts to specialize

**3. Scale exploded while active compute stayed manageable**

| Year | Total Params | Active Params | Ratio |
|------|-------------|---------------|-------|
| 2017 | 137B (Shazeer) | ~1B | ~0.7% |
| 2021 | 1.6T (Switch) | ~1.6B | ~0.1% |
| 2023 | 47B (Mixtral) | ~13B | ~28% |
| 2025 | 671B (DeepSeek-V3) | ~37B | ~5.5% |
| 2026 | ~27B (Gemma 4) | ~1.5B | ~5.6% |

### ⚡ Alternatives to Standard MoE

MoE isn't the only approach to conditional computation. Here are the main alternatives:

- **Sparse Attention** (e.g., Longformer, BigBird) — sparsity in the *attention* pattern rather than the FFN. Orthogonal to MoE and can be combined with it.
- **Early Exit** — tokens that are "easy" skip later layers entirely. Reduces compute for simple inputs but doesn't increase model capacity.
- **Mixture of Depths** (Raposo et al., 2024) — instead of routing tokens to different experts, route them to different *depths*. Some tokens skip entire transformer blocks.
- **Hybrid MoE + Dense** — alternate MoE layers with dense layers (used in Mixtral and Gemma 4). Not every layer needs to be MoE — typically every other layer is sufficient.

💡 The trend is clear: future models will combine multiple forms of conditional computation — MoE for capacity, early exit for efficiency, and mixture of depths for adaptive compute per token.

### 🎯 Interview Tip: The MoE Timeline

When asked about MoE in interviews, knowing the timeline signals deep understanding:

> *"MoE dates back to Jacobs et al. in 1991, but it took Shazeer's 2017 work to make it practical for NLP at scale. The Switch Transformer in 2021 simplified routing to K=1 and hit 1.6T parameters. Mixtral in 2023 was the first open-weight MoE LLM. DeepSeek-V3 in 2025 pushed to 671B params with 256 experts and introduced auxiliary-loss-free routing. Gemma 4 in 2026 adopted the shared expert pattern with 128 routed + 1 shared expert."*

Key numbers to memorize:
- **Mixtral**: N=8, K=2, 47B total, ~13B active
- **DeepSeek-V3**: N=256, K=8, 671B total, ~37B active, shared expert, no aux loss
- **Gemma 4**: N=128, K=2, shared expert

⚠️ Common interview mistake: confusing *total parameters* with *active parameters per token*. MoE models are often quoted by total params (671B for DeepSeek-V3), but the inference cost is determined by active params (~37B). Always clarify which number you're discussing.